<a href="https://colab.research.google.com/github/vcellmike/PatternsFormation/blob/main/Working/2024_08_21_XGBoost_on_ImageJ_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import json
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoModel, AutoTokenizer, get_scheduler
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from tqdm.notebook import tqdm, trange
from time import perf_counter
from PIL import Image
import pandas as pd
#from google.colab import drive
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle

print("All dependencies present.")

All dependencies present.


In [11]:
# set random seeds for repeatability
import numpy as np
import random

def set_seed(seed_val):
    random.seed(seed_val)
    np.random.seed(seed_val)
    torch.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)

print("Seed set")

Seed set


In [12]:
seed_val = 42
set_seed(seed_val)

In [13]:
file_in = "data/unclassified_features_39517"

with open(file_in + '.pkl', 'rb') as f:
    feats_df = pickle.load(f)

#load in dataframe
pd.set_option('display.max_columns', None)

file_out = file_in + "_s" + str(seed_val) + "_XGB" + ".model"

print(file_out)
print(feats_df.shape)

feats_df.head()


data/unclassified_features_39517_s42_XGB.model
(39517, 111)


,Ua,Ui,Ga,Gi,Ba,Da,Di,pattern,noise,path,seed,dir,feat_bool,num_spots,Mean,Median,Area_mean,Area_std,X_mean,X_std,Y_mean,Y_std,Perim._mean,Perim._std,BX_mean,BX_std,BY_mean,BY_std,Width_mean,Width_std,Height_mean,Height_std,Major_mean,Major_std,Minor_mean,Minor_std,Angle_mean,Angle_std,Circ._mean,Circ._std,Feret_mean,Feret_std,IntDen_mean,IntDen_std,%Area_mean,%Area_std,RawIntDen_mean,RawIntDen_std,FeretX_mean,FeretX_std,FeretY_mean,FeretY_std,FeretAngle_mean,FeretAngle_std,MinFeret_mean,MinFeret_std,AR_mean,AR_std,Round_mean,Round_std,Solidity_mean,Solidity_std,num_spots_inverted,Mean_inverted,Median_inverted,Area_inverted_mean,Area_inverted_std,X_inverted_mean,X_inverted_std,Y_inverted_mean,Y_inverted_std,Perim._inverted_mean,Perim._inverted_std,BX_inverted_mean,BX_inverted_std,BY_inverted_mean,BY_inverted_std,Width_inverted_mean,Width_inverted_std,Height_inverted_mean,Height_inverted_std,Major_inverted_mean,Major_inverted_std,Minor_inverted_mean,Minor_inverted_std,Angle_inverted_mean,Angle_inverted_std,Circ._inverted_mean,Circ._inverted_std,Feret_inverted_mean,Feret_inverted_std,IntDen_inverted_mean,IntDen_inverted_std,%Area_inverted_mean,%Area_inverted_std,RawIntDen_inverted_mean,RawIntDen_inverted_std,FeretX_inverted_mean,FeretX_inverted_std,FeretY_inverted_mean,FeretY_inverted_std,FeretAngle_inverted_mean,FeretAngle_inverted_std,MinFeret_inverted_mean,MinFeret_inverted_std,AR_inverted_mean,AR_inverted_std,Round_inverted_mean,Round_inverted_std,Solidity_inverted_mean,Solidity_inverted_std
0,0.011652,0.1066,0.115805,0.061599,-0.043244,0.007928,0.559698,0,2,0.png,0,/content/Images3/,1,1,255.000,255,40000.000000,0.000000,100.000000,0.000000,100.000000,0.000000,797.657000,0.000000,0.000000,0.000000,0.000000,0.000000,200.000000,0.000000,200.000000,0.000000,225.676000,0.000000,225.676000,0.000000,0.000000,0.000000,0.790000,0.000000,282.843000,0.000000,1.020000e+07,0.000000,100.0,0.0,1.020000e+07,0.000000,0.000000,0.000000,0.000000,0.000000,135.00000,0.000000,200.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0,0.000,0,40000.0,0.0,100.000,0.0,100.000,0.0,800.000,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,225.676,0.0,225.676,0.0,0.000,0.0,0.785,0.0,282.843,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,135.0,0.0,200.0,0.0,1.000,0.0,1.000,0.0,1.000,0.0
1,0.025122,0.063822,0.071957,0.106327,-0.113131,0.009186,0.611871,1,2,1.png,1,/content/Images3/,1,70,5.489,0,12.300000,2.548669,99.395943,58.506181,97.530229,58.913237,12.282143,1.400962,97.371429,58.453442,95.528571,58.843551,4.071429,0.723512,4.057143,0.753766,4.245571,0.439294,3.664200,0.511781,55.108900,57.413924,0.958686,0.069883,4.926700,0.412294,3.136500e+03,649.910626,100.0,0.0,3.136500e+03,649.910626,97.942857,58.572760,96.014286,58.843981,120.12900,30.775307,3.653171,0.564778,1.177686,0.183575,0.866686,0.115368,0.879486,0.067138,1,249.511,255,39139.0,0.0,99.998,0.0,100.022,0.0,811.882,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,223.351,0.0,223.117,0.0,38.796,0.0,0.746,0.0,282.843,0.0,9980445.0,0.0,100.0,0.0,9980445.0,0.0,0.0,0.0,200.0,0.0,45.0,0.0,200.0,0.0,1.001,0.0,0.999,0.0,0.979,0.0
2,0.014455,0.085124,0.091124,0.113974,-0.060333,0.001776,1.798215,0,2,2.png,2,/content/Images3/,1,1,255.000,255,40000.000000,0.000000,100.000000,0.000000,100.000000,0.000000,797.657000,0.000000,0.000000,0.000000,0.000000,0.000000,200.000000,0.000000,200.000000,0.000000,225.676000,0.000000,225.676000,0.000000,0.000000,0.000000,0.790000,0.000000,282.843000,0.000000,1.020000e+07,0.000000,100.0,0.0,1.020000e+07,0.000000,0.000000,0.000000,0.000000,0.000000,135.00000,0.000000,200.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0,0.000,0,40000.0,0.0,100.000,0.0,100.000,0.0,800.000,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,225.676,0.0,225.676,0.0,0.000,0.0,0.785,0.0,282.843,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,135.0,0.0,200.0,0.0,1.000,0.0,1.000,0.0,1.000,0.0
3,0.034475,0.059956,0.080798,0.10037,-0.132577,0.008852,0.854084,1,2,3.png,3,/content/Images3/,1,63,4

In [14]:
print(feats_df.shape)
feats_df.head()

(39517, 111)


,Ua,Ui,Ga,Gi,Ba,Da,Di,pattern,noise,path,seed,dir,feat_bool,num_spots,Mean,Median,Area_mean,Area_std,X_mean,X_std,Y_mean,Y_std,Perim._mean,Perim._std,BX_mean,BX_std,BY_mean,BY_std,Width_mean,Width_std,Height_mean,Height_std,Major_mean,Major_std,Minor_mean,Minor_std,Angle_mean,Angle_std,Circ._mean,Circ._std,Feret_mean,Feret_std,IntDen_mean,IntDen_std,%Area_mean,%Area_std,RawIntDen_mean,RawIntDen_std,FeretX_mean,FeretX_std,FeretY_mean,FeretY_std,FeretAngle_mean,FeretAngle_std,MinFeret_mean,MinFeret_std,AR_mean,AR_std,Round_mean,Round_std,Solidity_mean,Solidity_std,num_spots_inverted,Mean_inverted,Median_inverted,Area_inverted_mean,Area_inverted_std,X_inverted_mean,X_inverted_std,Y_inverted_mean,Y_inverted_std,Perim._inverted_mean,Perim._inverted_std,BX_inverted_mean,BX_inverted_std,BY_inverted_mean,BY_inverted_std,Width_inverted_mean,Width_inverted_std,Height_inverted_mean,Height_inverted_std,Major_inverted_mean,Major_inverted_std,Minor_inverted_mean,Minor_inverted_std,Angle_inverted_mean,Angle_inverted_std,Circ._inverted_mean,Circ._inverted_std,Feret_inverted_mean,Feret_inverted_std,IntDen_inverted_mean,IntDen_inverted_std,%Area_inverted_mean,%Area_inverted_std,RawIntDen_inverted_mean,RawIntDen_inverted_std,FeretX_inverted_mean,FeretX_inverted_std,FeretY_inverted_mean,FeretY_inverted_std,FeretAngle_inverted_mean,FeretAngle_inverted_std,MinFeret_inverted_mean,MinFeret_inverted_std,AR_inverted_mean,AR_inverted_std,Round_inverted_mean,Round_inverted_std,Solidity_inverted_mean,Solidity_inverted_std
0,0.011652,0.1066,0.115805,0.061599,-0.043244,0.007928,0.559698,0,2,0.png,0,/content/Images3/,1,1,255.000,255,40000.000000,0.000000,100.000000,0.000000,100.000000,0.000000,797.657000,0.000000,0.000000,0.000000,0.000000,0.000000,200.000000,0.000000,200.000000,0.000000,225.676000,0.000000,225.676000,0.000000,0.000000,0.000000,0.790000,0.000000,282.843000,0.000000,1.020000e+07,0.000000,100.0,0.0,1.020000e+07,0.000000,0.000000,0.000000,0.000000,0.000000,135.00000,0.000000,200.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0,0.000,0,40000.0,0.0,100.000,0.0,100.000,0.0,800.000,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,225.676,0.0,225.676,0.0,0.000,0.0,0.785,0.0,282.843,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,135.0,0.0,200.0,0.0,1.000,0.0,1.000,0.0,1.000,0.0
1,0.025122,0.063822,0.071957,0.106327,-0.113131,0.009186,0.611871,1,2,1.png,1,/content/Images3/,1,70,5.489,0,12.300000,2.548669,99.395943,58.506181,97.530229,58.913237,12.282143,1.400962,97.371429,58.453442,95.528571,58.843551,4.071429,0.723512,4.057143,0.753766,4.245571,0.439294,3.664200,0.511781,55.108900,57.413924,0.958686,0.069883,4.926700,0.412294,3.136500e+03,649.910626,100.0,0.0,3.136500e+03,649.910626,97.942857,58.572760,96.014286,58.843981,120.12900,30.775307,3.653171,0.564778,1.177686,0.183575,0.866686,0.115368,0.879486,0.067138,1,249.511,255,39139.0,0.0,99.998,0.0,100.022,0.0,811.882,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,223.351,0.0,223.117,0.0,38.796,0.0,0.746,0.0,282.843,0.0,9980445.0,0.0,100.0,0.0,9980445.0,0.0,0.0,0.0,200.0,0.0,45.0,0.0,200.0,0.0,1.001,0.0,0.999,0.0,0.979,0.0
2,0.014455,0.085124,0.091124,0.113974,-0.060333,0.001776,1.798215,0,2,2.png,2,/content/Images3/,1,1,255.000,255,40000.000000,0.000000,100.000000,0.000000,100.000000,0.000000,797.657000,0.000000,0.000000,0.000000,0.000000,0.000000,200.000000,0.000000,200.000000,0.000000,225.676000,0.000000,225.676000,0.000000,0.000000,0.000000,0.790000,0.000000,282.843000,0.000000,1.020000e+07,0.000000,100.0,0.0,1.020000e+07,0.000000,0.000000,0.000000,0.000000,0.000000,135.00000,0.000000,200.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0,0.000,0,40000.0,0.0,100.000,0.0,100.000,0.0,800.000,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,225.676,0.0,225.676,0.0,0.000,0.0,0.785,0.0,282.843,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,135.0,0.0,200.0,0.0,1.000,0.0,1.000,0.0,1.000,0.0
3,0.034475,0.059956,0.080798,0.10037,-0.132577,0.008852,0.854084,1,2,3.png,3,/content/Images3/,1,63,4

In [15]:
print(feats_df.columns[13:111])

Index(['num_spots', 'Mean', 'Median', 'Area_mean', 'Area_std', 'X_mean',
       'X_std', 'Y_mean', 'Y_std', 'Perim._mean', 'Perim._std', 'BX_mean',
       'BX_std', 'BY_mean', 'BY_std', 'Width_mean', 'Width_std', 'Height_mean',
       'Height_std', 'Major_mean', 'Major_std', 'Minor_mean', 'Minor_std',
       'Angle_mean', 'Angle_std', 'Circ._mean', 'Circ._std', 'Feret_mean',
       'Feret_std', 'IntDen_mean', 'IntDen_std', '%Area_mean', '%Area_std',
       'RawIntDen_mean', 'RawIntDen_std', 'FeretX_mean', 'FeretX_std',
       'FeretY_mean', 'FeretY_std', 'FeretAngle_mean', 'FeretAngle_std',
       'MinFeret_mean', 'MinFeret_std', 'AR_mean', 'AR_std', 'Round_mean',
       'Round_std', 'Solidity_mean', 'Solidity_std', 'num_spots_inverted',
       'Mean_inverted', 'Median_inverted', 'Area_inverted_mean',
       'Area_inverted_std', 'X_inverted_mean', 'X_inverted_std',
       'Y_inverted_mean', 'Y_inverted_std', 'Perim._inverted_mean',
       'Perim._inverted_std', 'BX_inverted_mean', 'BX_

XGBOOST

In [16]:
#! pip install xgboost
#! pip install openpyxl
#! pip install tensorflow
#! pip install graphviz
#! pip install hyperopt

from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
import numpy as np
from sklearn.model_selection import RepeatedKFold
from numpy import absolute
from pandas import read_csv
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from xgboost import XGBRegressor
import openpyxl
from xgboost import cv
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import operator
# for loading/processing the images
import tensorflow
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
from keras.applications.vgg16 import preprocess_input

# models
from keras.applications.vgg16 import VGG16
from keras.models import Model

# clustering and dimension reduction
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn import tree
import graphviz
from sklearn import metrics
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
from hyperopt import fmin, tpe, hp,STATUS_OK
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

print("All dependencies present")

All dependencies present


In [17]:
# scale data 
# uses Standard Scaler, range [-1, 1]

## X (independent variable) --> the feature values themselves
X = feats_df[feats_df.columns[13:111]].astype(float)

## Y (dependent variables) --> the parameter values of Negan and RTO themselves
y = feats_df[["Ua","Ui","Ga","Gi","Da","Di","Ba"]].astype(float)

#Scale data with standardscaler
scaling = StandardScaler()

# Use fit and transform method
scaling.fit(X)
X_scaled = scaling.transform(X)

# select 20 percent for testing
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42) #create training split


In [18]:
print(X_test)
#print(y_train)

[[ 0.84427681 -0.56527056 -0.81564454 ... -0.30076773 -0.9992852
  -0.26485482]
 [ 1.04601638  0.03622808 -0.81564454 ... -0.30076773 -3.22997976
  -0.26485482]
 [-0.46172146  1.28776123  1.22602426 ... -0.30076773  0.56389092
  -0.26485482]
 ...
 [-0.47233933 -0.98652117 -0.81564454 ... -0.30076773  0.56389092
  -0.26485482]
 [-0.47233933 -0.98652117 -0.81564454 ... -0.30076773  0.56389092
  -0.26485482]
 [-0.42986785  0.56368322  1.22602426 ...  3.92583447 -0.52963874
   2.96037405]]


In [19]:
y_test

,Ua,Ui,Ga,Gi,Da,Di,Ba
27444,0.034815,0.086269,0.098214,0.111581,0.010052,1.299758,-0.097848
11068,0.030000,0.240000,0.075000,0.100000,0.013000,0.500000,-0.120000
21463,0.015112,0.146368,0.100549,0.063209,0.010806,0.530263,-0.178364
31013,0.027648,0.159852,0.166757,0.120432,0.011730,1.441355,-0.128601
16157,0.014156,0.136370,0.043016,0.117460,0.008501,0.260931,-0.058947
...,...,...,...,...,...,...,...
27390,0.050426,0.077152,0.085455,0.127332,0.011023,0.739167,-0.123279
3082,0.050221,0.165615,0.046623,0.175211,0.010444,1.380281,-0.058233
37277,0.034758,0.025755,0.161337,0.160499,0.011274,0.992567,-0.125318
31798,0.046418,0.174132,0.050813,0.173091,0.009446,0.788921,-0.105999


In [20]:
model = xgb.XGBRegressor(n_estimators=1000, max_depth=10, eta=0.1, subsample=0.7, colsample_bytree=0.8, num_boost_round=50, objective= "reg:squarederror", device = "cuda")
model.fit(X_train, y_train)

/Users/mikhailblinov/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [14:27:18] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)
/Users/mikhailblinov/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [14:27:18] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "num_boost_round" } are not used.

  warnings.warn(smsg, UserWarning)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eta=0.1, eval_metric=None,
             feature_types=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=None, num_boost_round=50, ...)

In [21]:
## Save XGBoost Model to File
model.save_model(file_out)

/Users/mikhailblinov/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [14:27:51] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)


In [22]:
loaded_model = xgb.XGBRegressor()

loaded_model.load_model(file_out)

#loaded_model.fit(X_train, y_train)

# make predictions
y_pred = loaded_model.predict(X_test)

pd.DataFrame(data=y_pred) 

# Columns - parameters 1-7
# Rows - Subsample of rows

print("Model loaded")

Model loaded


In [23]:
# GOAL: To determine the error rate in predicting the value of continuous, parameter data

# Converts test Y values to numpy array
true_vals = np.array(y_test)

# Outputs amount of true values
print(true_vals.shape)

# Outputs amount of predicted values
print(y_pred.shape)

# Sets counters for correct, incorrect, errors numpy array
correct = 0
incorrect = 0
p_errs = np.zeros(7)

# loops through each row in the true_vals array
for i in range(true_vals.shape[0]):
  # predicted value = predicted value from loop
  pred = y_pred[i]
  # true value = true value from loop
  true_val = true_vals[i]
  # Adds to error: absolute percent difference between true and predicted values
  p_errs += (np.abs((pred-true_val)/true_val))

# Outputs error percentages
print((p_errs/true_vals.shape[0])*100)

(7904, 7)
(7904, 7)
[146.50361786 120.20670433  87.90838201  72.57677275          inf
  53.51171384  56.57515089]


/var/folders/3p/pbgs38h94vdd8kjwbd33x40h0000gr/T/ipykernel_19280/1689613994.py:24: RuntimeWarning: divide by zero encountered in divide
  p_errs += (np.abs((pred-true_val)/true_val))


In [26]:
# select features of real images

new_df = pd.read_pickle("data/real_df_xgboost.pkl")
new_feats = new_df[new_df.columns[3:111]]

# new_df = pd.read_pickle("./data/Images_Classified_np126.pkl")
# new_feats = new_df[new_df.columns[13:111]]

print(new_feats.head())

y_pred = None

loaded_model = xgb.XGBRegressor()

loaded_model.load_model(file_out)


#scale data
scaling=StandardScaler()

# Use fit and transform method
scaling.fit(new_feats)
new_feats_scaled = scaling.transform(new_feats)
print(new_feats_scaled.shape)

# predict params of images from real feats
y_pred = loaded_model.predict(new_feats_scaled)
print(y_pred[2])

new_df["pred_params"] = list(y_pred)

   num_spots     Mean  Median     Area_mean      Area_std      X_mean  \
0        101    5.699       0      8.851485     11.066304  125.673327   
1         49   23.479       0     75.163265    103.827240   99.147755   
2          2  251.322     255  19711.500000  19688.500000  148.632000   
3         77   15.306       0     31.181818     46.025890  101.898026   
4         32   62.124       0    304.531250    876.516507  104.642344   

       X_std      Y_mean      Y_std  Perim._mean  Perim._std     BX_mean  \
0  57.972907   88.902515  52.059334    10.451525    8.653331  123.960396   
1  55.940428   74.931612  51.325268    36.572388   37.781366   95.387755   
2  49.216000   51.684500  49.271500   437.255000  417.941000   97.000000   
3  40.868593  104.009818  57.481703    19.575195   18.250837   99.701299   
4  49.916328   82.908906  45.964830    98.939469  229.053596   97.843750   

      BX_std     BY_mean     BY_std  Width_mean  Width_std  Height_mean  \
0  57.832452   87.297030  52.

In [27]:
new_df

,path,feats_df,inverse_feats_df,num_spots,Mean,Median,Area_mean,Area_std,X_mean,X_std,Y_mean,Y_std,Perim._mean,Perim._std,BX_mean,BX_std,BY_mean,BY_std,Width_mean,Width_std,Height_mean,Height_std,Major_mean,Major_std,Minor_mean,Minor_std,Angle_mean,Angle_std,Circ._mean,Circ._std,Feret_mean,Feret_std,IntDen_mean,IntDen_std,%Area_mean,%Area_std,RawIntDen_mean,RawIntDen_std,FeretX_mean,FeretX_std,FeretY_mean,FeretY_std,FeretAngle_mean,FeretAngle_std,MinFeret_mean,MinFeret_std,AR_mean,AR_std,Round_mean,Round_std,Solidity_mean,Solidity_std,num_spots_inverted,Mean_inverted,Median_inverted,Area_inverted_mean,Area_inverted_std,X_inverted_mean,X_inverted_std,Y_inverted_mean,Y_inverted_std,Perim._inverted_mean,Perim._inverted_std,BX_inverted_mean,BX_inverted_std,BY_inverted_mean,BY_inverted_std,Width_inverted_mean,Width_inverted_std,Height_inverted_mean,Height_inverted_std,Major_inverted_mean,Major_inverted_std,Minor_inverted_mean,Minor_inverted_std,Angle_inverted_mean,Angle_inverted_std,Circ._inverted_mean,Circ._inverted_std,Feret_inverted_mean,Feret_inverted_std,IntDen_inverted_mean,IntDen_inverted_std,%Area_inverted_mean,%Area_inverted_std,RawIntDen_inverted_mean,RawIntDen_inverted_std,FeretX_inverted_mean,FeretX_inverted_std,FeretY_inverted_mean,FeretY_inverted_std,FeretAngle_inverted_mean,FeretAngle_inverted_std,MinFeret_inverted_mean,MinFeret_inverted_std,AR_inverted_mean,AR_inverted_std,Round_inverted_mean,Round_inverted_std,Solidity_inverted_mean,Solidity_inverted_std,pred_params
0,LF10_11-20-23_(green)_(0-152).tif.csv,Label ...,Label Area...,101,5.699,0,8.851485,11.066304,125.673327,57.972907,88.902515,52.059334,10.451525,8.653331,123.960396,57.832452,87.297030,52.027705,3.435644,2.619107,3.188119,2.423944,3.860287,2.502460,2.196881,1.250776,50.320287,53.900986,0.857059,0.194583,4.505861,3.106018,2.257129e+03,2.821908e+03,100.0,0.0,2.257129e+03,2.821908e+03,124.089109,57.843781,88.920792,51.956509,103.979446,47.466852,2.599564,1.875266,1.738505,0.579683,0.643267,0.218933,0.857396,0.144207,2,249.301,255,19553.000000,19551.000000,149.373000,50.127000,81.072500,19.072500,405.556000,400.728000,99.500000,99.500000,30.500000,30.500000,100.500000,99.500000,101.000000,99.000000,113.250500,110.993500,111.578500,110.450500,105.140500,15.140500,0.878000,0.122000,142.539500,140.303500,4.986015e+06,4.985505e+06,100.0,0.0,4.986015e+06,4.985505e+06,99.500000,99.500000,30.500000,30.500000,125.782500,9.217500,100.500000,99.500000,1.505000,0.495000,0.745000,0.245000,0.989000,0.011000,"[0.07444812, 0.1587155, 0.16584946, 0.14243883..."
1,mpar_rto(c-c)_(green)_(0-137).tif.csv,Label Ar...,Label Area...,49,23.479,0,75.163265,103.827240,99.147755,55.940428,74.931612,51.325268,36.572388,37.781366,95.387755,55.720445,68.224490,47.695576,7.571429,6.676184,13.428571,13.490738,12.331796,11.379018,5.113531,3.973676,73.345184,38.726102,0.654939,0.264055,14.381224,13.651513,1.916663e+04,2.647595e+04,100.0,0.0,1.916663e+04,2.647595e+04,96.469388,56.061956,74.387755,52.484329,98.199592,32.841924,6.559959,5.483810,2.445878,1.308504,0.527082,0.253666,0.789898,0.171918,1,231.521,255,36317.000000,0.000000,99.125000,0.000000,98.020000,0.000000,810.728000,0.000000,0.000000,0.000000,0.000000,0.000000,200.000000,0.000000,200.000000,0.000000,216.098000,0.000000,213.978000,0.000000,6.379000,0.000000,0.694000,0.000000,282.843000,0.000000,9.260835e+06,0.000000e+00,100.0,0.0,9.260835e+06,0.000000e+00,0.000000,0.000000,0.000000,0.000000,135.000000,0.000000,200.000000,0.000000,1.010000,0.000000,0.990000,0.000000,0.908000,0.000000,"[0.07012652, 0.16461392, 0.16871558, 0.1352927..."
2,mpar_rto_crispr_(green)_(0-155).tif.csv,Label Ar...,Label Ar...,2,251.322,255,19711.500000,19688.500000,148.632000,49.216000,51.684500,49.271500,437.255000,417.941000,97.000000,97.000000,0.000000,0.000000,103.000000,97.000000,103.000000,97.000000,115.674500,108.751500,113.879500,109.649500,124.750000,2.012000,0.726000,0.049000,145.664000,137.179000,5.026432e+06,5.020568e+06,100.0,0.0,5.026432e+06

In [32]:
## TODO: Grab dataframe at the top (w/ all the features), drop all unnessesary columns (diff. from real_df, drop parameters)
## choose first five images and run it and see how close they are. 

print ("Ua","Ui","Ga","Gi","Da","Di","Ba")
for i in range(new_df.shape[0]):
  print(new_df["path"][i])
  print(new_df["pred_params"][i])
  #print(new_df["Ua"][i], new_df["Ui"][i],new_df["Ga"][i], new_df["Gi"][i],new_df["Da"][i], new_df["Di"][i],new_df["Ba"][i]
     


Ua Ui Ga Gi Da Di Ba
LF10_11-20-23_(green)_(0-152).tif.csv
[ 0.07444812  0.1587155   0.16584946  0.14243883  0.04621829  0.7816448
 -0.12434022]
mpar_rto(c-c)_(green)_(0-137).tif.csv
[ 0.07012652  0.16461392  0.16871558  0.13529272  0.04601083  0.8899547
 -0.119705  ]
mpar_rto_crispr_(green)_(0-155).tif.csv
[ 0.14765821  0.2103977   0.19482994  0.24405631  0.14133334 -2.6961794
 -0.2267239 ]
MLC_F1_11-20-23_(green)_(0-105).tif.csv
[ 0.07204828  0.17388527  0.1414945   0.13030724  0.04593876  4.1902494
 -0.1415103 ]
RTO_rnai_med_(green)_(0-109).tif.csv
[ 0.07386646  0.09276281  0.14801314  0.10387754  0.04547361  1.1544236
 -0.13415183]
NEGAN_crispr_(green)_(0-150).tif.csv
[ 0.14634633  0.21512721  0.19381776  0.2353293   0.14035179  4.2921557
 -0.25183842]
mpar_wt_(green)_(0-170).tif.csv
[ 0.0768711   0.16447178  0.15797749  0.1304044   0.04629316  3.3565116
 -0.14158332]
RTO_rnai_low_(green)_(0-115).tif.csv
[ 0.06952988  0.16020426  0.13923697  0.14415951  0.04632314  2.4976082
 -0.13